In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
!pip install gdown

In [3]:
import gdown
id = "1BZSSltHaBW0KK0tVxK43F3vv5rqgW1c5"
gdown.download(id = id, output="/content/low_conf_sample.parquet")

Downloading...
From: https://drive.google.com/uc?id=1BZSSltHaBW0KK0tVxK43F3vv5rqgW1c5
To: /content/low_conf_sample.parquet
100%|██████████| 5.31M/5.31M [00:00<00:00, 20.1MB/s]


'/content/low_conf_sample.parquet'

In [4]:
!wget https://raw.githubusercontent.com/TranTheHung2312332/FbOM-from-amazon-ds-PTIT/main/scripts/analyze_aspect_sentiment.py

--2026-05-20 05:59:59--  https://raw.githubusercontent.com/TranTheHung2312332/FbOM-from-amazon-ds-PTIT/main/scripts/analyze_aspect_sentiment.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3595 (3.5K) [text/plain]
Saving to: ‘analyze_aspect_sentiment.py’

analyze_aspect_sent 100%[===================>]   3.51K  --.-KB/s    in 0s      

2026-05-20 06:00:00 (65.2 MB/s) - ‘analyze_aspect_sentiment.py’ saved [3595/3595]



In [5]:
import os
import re
import ast
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [6]:
import importlib
import analyze_aspect_sentiment
importlib.reload(analyze_aspect_sentiment)
from analyze_aspect_sentiment import predict_asc,asc_tokenizer,asc_model

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [7]:
POLAR_THRESHOLD = 0.55
NEUTRAL_THRESHOLD = 0.9

asc_model_path = "/content/drive/MyDrive/absa_self_train_phase1/asc_teacher_phase1"
indir = "/content/low_conf_sample.parquet"
savedir = "/content/drive/MyDrive/data_train/asc"

In [ ]:
# @title
df_ate = pd.read_parquet(indir)
df_ate

,parent_asin,sentence_id,sentence_text,rating,category_name,gate_confidence,aspects,confidences
0,B004XMHX9W,11,five [GENERIC_NOUN] very good can read my loca...,5.0,Software,0.009793,[],[]
1,B06XFJQSYW,499,last [GENERIC_NOUN] no [GENERIC_NOUN] so what ...,1.0,Software,0.005684,[],[]
2,B0088QGB7Y,4964,we compete to see which [GENERIC_NOUN] of us c...,5.0,Software,0.005602,[],[]
3,B00K7WGUKA,3420,requires no wifi and is a great [GENERIC_NOUN]...,5.0,Software,0.930865,[wifi],[0.9334377646446228]
4,B01LXZLEMN,4740,the cw advert showed several series i wanted t...,2.0,Software,0.969513,[cw advert],[0.9300174415111542]
...,...,...,...,...,...,...,...,...
3808772,B006HJKKCG,735,actually speaks the words,5.0,Software,0.006586,[],[]
3808773,B0051HRMNU,53,cool this app is really useful,5.0,Software,0.984674,[app],[0.9943851232528687]
3808774,B01LXOU5PM,1739,does not work on fire this app does not work o...,1.0,Software,0.987768,[app],[0.9814614653587341]
3808775,B000077DD8,146,of course youll have the ser and estar verbs d...,1.0,Software,0.009863,[],[]


In [8]:
df_low = pd.read_parquet(indir)
df_low

,parent_asin,sentence_id,sentence_text,rating,category_name,aspects
0,0072119527,8,since the test is becoming adaptive the next e...,5.0,Software,"[test, learning tool]"
1,0321700945,139,this really isnt a [GENERIC_NOUN] since the le...,4.0,Software,"[lessons, lesson]"
2,0321700945,19,good class for beginners not as useful for exi...,3.0,Software,[class]
3,0321700945,26,i wish however they just made a chapter that l...,3.0,Software,[chapter]
4,0321700945,55,and while lightroom isnt too hard to figure ou...,5.0,Software,[lightroom]
...,...,...,...,...,...,...
79156,B0C8GBSN27,38,br dld wo incident supplied [GENERIC_NOUN] act...,1.0,Software,[activation code entry]
79157,B0C8GBSN27,5,the ok button is grey,3.0,Software,[ok button]
79158,B0C9BBDBPM,141,the installation took more [GENERIC_NOUN] than...,5.0,Software,[installation]
79159,B0C9BBDBPM,175,it has a password manager feature that avast d...,5.0,Software,[password manager feature]


In [12]:
expanded_rows = []
for _, row in tqdm(
    df_low.iterrows(),
    total=len(df_low),
):
    aspects = row["aspects"]
    for aspect in aspects:
        expanded_rows.append({
            "parent_asin": row["parent_asin"],
            "sentence_id": row["sentence_id"],
            "sentence_text": row["sentence_text"],
            "rating": row["rating"],
            "category_name": row["category_name"],
            "aspects": aspect,
        })
expanded_df = pd.DataFrame(
    expanded_rows
)

100%|██████████| 79161/79161 [00:03<00:00, 21149.21it/s]


In [35]:
expanded_df

,parent_asin,sentence_id,sentence_text,rating,category_name,aspect
0,0072119527,8,since the test is becoming adaptive the next e...,5.0,Software,test
1,0072119527,8,since the test is becoming adaptive the next e...,5.0,Software,learning tool
2,0321700945,139,this really isnt a [GENERIC_NOUN] since the le...,4.0,Software,lessons
3,0321700945,139,this really isnt a [GENERIC_NOUN] since the le...,4.0,Software,lesson
4,0321700945,19,good class for beginners not as useful for exi...,3.0,Software,class
...,...,...,...,...,...,...
116622,B0C8GBSN27,38,br dld wo incident supplied [GENERIC_NOUN] act...,1.0,Software,activation code entry
116623,B0C8GBSN27,5,the ok button is grey,3.0,Software,ok button
116624,B0C9BBDBPM,141,the installation took more [GENERIC_NOUN] than...,5.0,Software,installation
116625,B0C9BBDBPM,175,it has a password manager feature that avast d...,5.0,Software,password manager feature


In [10]:
predictions = predict_asc(
    expanded_df["sentence_text"],
    expanded_df["aspect"],
    polar_threshold = POLAR_THRESHOLD,
    neutral_threshold = NEUTRAL_THRESHOLD
)

100%|██████████| 1823/1823 [09:54<00:00,  3.07it/s]


In [11]:
import json
with open('/content/drive/MyDrive/data_train/asc/asc_data/predictions_low_confident.json', 'w') as f:
    json.dump(predictions, f)

In [10]:
import json
with open('/content/drive/MyDrive/data_train/asc/asc_data/predictions_low_confident.json', 'r') as f:
    predictions = json.load(f)

In [13]:
expanded_df["sentiments"] = [
    x[0]
    for x in predictions
]
expanded_df["is_high"] = [
    x[1]
    for x in predictions
]

In [15]:
# SPLIT
high_df = expanded_df[
    expanded_df["is_high"] == True
].drop(columns=["is_high"])

low_df = expanded_df[
    expanded_df["is_high"] == False
][[
    "parent_asin",
    "sentence_id",
    "sentence_text",
    "rating",
    "category_name",
    "aspects",
]]

In [16]:
high_df

,parent_asin,sentence_id,sentence_text,rating,category_name,aspects,sentiments
1,0072119527,8,since the test is becoming adaptive the next e...,5.0,Software,learning tool,"[0.005856213625520468, 0.004701403900980949, 0..."
2,0321700945,139,this really isnt a [GENERIC_NOUN] since the le...,4.0,Software,lessons,"[0.21972425282001495, 0.17992304265499115, 0.6..."
11,0321719824,36,for the most [GENERIC_NOUN] the test questions...,4.0,Software,supplement,"[0.8611599206924438, 0.09174538403749466, 0.04..."
12,0321719824,36,for the most [GENERIC_NOUN] the test questions...,4.0,Software,video lessons,"[0.7030096650123596, 0.1717446744441986, 0.125..."
13,0321719824,36,for the most [GENERIC_NOUN] the test questions...,4.0,Software,chapters,"[0.9101657271385193, 0.06038719415664673, 0.02..."
...,...,...,...,...,...,...,...
116616,B0C8GBSN27,14,the photo and video editing choices are very l...,1.0,Software,photo and,"[0.9394806027412415, 0.04014025628566742, 0.02..."
116617,B0C8GBSN27,14,the photo and video editing choices are very l...,1.0,Software,video editing choices,"[0.945523202419281, 0.03487614542245865, 0.019..."
116618,B0C8GBSN27,14,the photo and video editing choices are very l...,1.0,Software,program,"[0.921947717666626, 0.040501415729522705, 0.03..."
116619,B0C8GBSN27,14,the photo and video editing choices are very l...,1.0,Software,install process,"[0.921947717666626, 0.040501415729522705, 0.03..."


In [48]:
old_high_df = pd.read_parquet("/content/drive/MyDrive/data_train/asc/p6/high_conf_sample.parquet")

In [46]:
def expand_asc_rows(df):
    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Expanding"):
        aspects = (row["aspects"])
        sentiments = (row["sentiments"])

        if len(aspects) != len(sentiments):
            raise ValueError(
                f"Mismatch aspects/sentiments length at sentence_id={row.get('sentence_id')}: "
                f"{len(aspects)} vs {len(sentiments)}"
            )
        for aspect, triplet in zip(aspects, sentiments):
            triplet = list(map(float, triplet))
            rows.append({
                "parent_asin": row["parent_asin"],
                "sentence_id": row["sentence_id"],
                "sentence_text": row["sentence_text"],
                "rating": row["rating"],
                "category_name": row["category_name"],
                "aspects": aspect,
                "sentiments": triplet,
            })

    return pd.DataFrame(rows)

In [49]:
old_high_df_expanded = expand_asc_rows(old_high_df.drop("gate_confidence",axis=1))

Expanding: 100%|██████████| 1955412/1955412 [02:06<00:00, 15442.24it/s]


In [62]:
new_high_df = pd.concat([old_high_df_expanded, high_df], ignore_index=True)

In [63]:
new_high_df.to_parquet(
    f"{savedir}/p8/high_conf_sample.parquet",
    index=False,
)
print(len(new_high_df))

2519547


In [28]:
high_df.to_parquet(
    f"{savedir}/high_conf_sample/high_conf_sample.parquet",
    index=False,
)
low_df.to_parquet(
    f"{savedir}/low_conf_sample/low_conf_sample.parquet",
    index=False,
)
print("High:", len(high_df))
print("Low :", len(low_df))

High: 33271
Low : 83356


In [60]:
total_samples = len(expanded_df)
high_conf_ratio = (
    len(high_df) / total_samples
    if total_samples > 0
    else 0
)
all_sentiments = []
for sentiments in high_df["sentiments"]:
    all_sentiments.extend([sentiments])

all_sentiments = np.array(all_sentiments)

pred_ids = np.argmax(
    all_sentiments,
    axis = 1
)
neg_ratio = np.mean(pred_ids == 0)
neu_ratio = np.mean(pred_ids == 1)
pos_ratio = np.mean(pred_ids == 2)

report_lines = []
report_lines.append(
    f"Category: Software"
)
report_lines.append(
    f"Total samples: {total_samples}"
)
report_lines.append(
    f"High confidence samples: {len(high_df)}"
)
report_lines.append(
    f"Low confidence samples: {len(low_df)}"
)
report_lines.append("")
report_lines.append(
    f"High confidence ratio: "
    f"{high_conf_ratio:.4f}"
)
report_lines.append("")
report_lines.append(
    "Sentiment distribution (high confidence):"
)
report_lines.append(
    f"Negative ratio: {neg_ratio:.4f}"
)
report_lines.append(
    f"Neutral ratio: {neu_ratio:.4f}"
)
report_lines.append(
    f"Positive ratio: {pos_ratio:.4f}"
)
report_text = "\n".join(report_lines)

report_path = (
    f"{savedir}/p8/"
    f"asc_2_report_Software.txt"
)
with open(
    report_path,
    "w",
    encoding="utf-8",
) as f:
    f.write(report_text)

print(report_text)

Category: Software
Total samples: 116627
High confidence samples: 33271
Low confidence samples: 83356

High confidence ratio: 0.2853

Sentiment distribution (high confidence):
Negative ratio: 0.4871
Neutral ratio: 0.0000
Positive ratio: 0.5129
